In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.cluster import KMeans

In [2]:
df = pd.read_csv("dataset_with_latlon_full.csv")
df.columns = df.columns.str.strip()

In [3]:
import re

def convert_price(price):
    try:
        price = str(price).replace("₹", "").replace(",", "").strip().lower()
        
        # remove weird text like "acs"
        price = re.sub(r"[a-zA-Z]+", "", price).strip()
        
        if "cr" in str(price).lower():
            return float(price.replace("cr", "")) * 1e7
        elif "l" in str(price).lower():
            return float(price.replace("l", "")) * 1e5
        else:
            return float(price)
    
    except:
        return None  

In [4]:
df["Price"] = df["Price"].apply(convert_price)


In [5]:
df["log_price"] = np.log1p(df["Price"]) #log transofrmation

# Convert Balcony to numeric
df["Balcony"] = df["Balcony"].map({"Yes":1, "No":0}).fillna(0)


df["area_per_bath"] = df["Total_Area"] / (df["Baths"] + 1) # Feature engineering

In [6]:
import re

def extract_bhk(title):
    try:
        match = re.search(r"(\d+)\s*BHK", str(title))
        if match:
            return int(match.group(1))
        else:
            return None
    except:
        return None

df["bhk"] = df["Property Title"].apply(extract_bhk)

In [7]:
print(df[["Property Title", "bhk"]].head(10))

                                      Property Title   bhk
0  4 BHK Flat for sale in Kanathur Reddikuppam, C...   4.0
1  10 BHK Independent House for sale in Pozhichal...  10.0
2      3 BHK Flat for sale in West Tambaram, Chennai   3.0
3  7 BHK Independent House for sale in Triplicane...   7.0
4              2 BHK Flat for sale in Avadi, Chennai   2.0
5           2 BHK Flat for sale in Siruseri, Chennai   2.0
6          2 BHK Flat for sale in Sembakkam, Chennai   2.0
7  3 BHK Independent House for sale in Mahindra W...   3.0
8      2 BHK Flat for sale in West Tambaram, Chennai   2.0
9          1 BHK Flat for sale in Chromepet, Chennai   1.0


In [8]:
df["rooms"] = df["Baths"] + 1   # rough proxy
df["log_area"] = np.log1p(df["Total_Area"])

In [9]:
df["bhk"] = df["bhk"].fillna(df["Baths"])  # handles the missing values

In [10]:
features = [
    "Total_Area",
    "Baths",
    "Balcony",
    "latitude",
    "longitude",
    "area_per_bath",
    
    "log_area",
    "bhk"
]
X = df[features]
y = df["log_price"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=800,
             n_jobs=None, num_parallel_tree=None, ...)

In [13]:
y_pred = model.predict(X_test)

In [14]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("BASELINE RESULTS")
print("RMSE:", rmse)
print("R2:", r2)

BASELINE RESULTS
RMSE: 1.0387734843449994
R2: 0.39122934156502565


In [15]:
kmeans = KMeans(n_clusters=50, random_state=42)
df["location_cluster"] = kmeans.fit_predict(df[["latitude","longitude"]])

In [16]:
df["area_per_bhk"] = df["Total_Area"] / (df["bhk"] + 1)
df["bath_per_bhk"] = df["Baths"] / (df["bhk"] + 1)

In [17]:
df["log_area"] = np.log1p(df["Total_Area"])

In [18]:
features = [
    "Total_Area",
    "Baths",
    "Balcony",
    "latitude",
    "longitude",
    "area_per_bath",
    
    "log_area",
    "bhk",
    "bath_per_bhk"
]
X = df[features]
y = df["log_price"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [21]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, ...)

In [22]:
y_pred = model.predict(X_test)

In [23]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("BASELINE RESULTS")
print("RMSE:", rmse)
print("R2:", r2)

BASELINE RESULTS
RMSE: 1.0451882181029863
R2: 0.38368744849202874
